# MiSTR quickstart

<a href="https://colab.research.google.com/github/timgzhou/MiSTR/blob/main/notebooks/mistr_quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Crop-type mapping from a year of Sentinel-2 and Sentinel-1 with the released MiSTR model
(OlmoEarth-Tiny, fine-tuned on PASTIS). MiSTR encodes the whole year at coarse resolution, then
re-encodes only the uncertain places, in the season that matters, at fine resolution.

Runs on CPU or any GPU (in Colab: *Runtime > Change runtime type > T4 GPU* makes it faster).
Just run all cells.

In [ ]:
%pip install -q "mistr @ git+https://github.com/timgzhou/MiSTR"

## Load the model and a sample
The model (~26 MB) and two PASTIS test patches (128 x 128 px at 10 m) come from the Hugging Face Hub.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from huggingface_hub import hf_hub_download
from mistr import MiSTR, prepare

REPO = "TimZhou/mistr-olmoearth-tiny-pastis-s6s3"
model = MiSTR.from_pretrained(REPO)
names = model.config["class_names"]
print("device:", model.device, "| attention:", "FlashAttention" if model.flash else "PyTorch SDPA")

z = np.load(hf_hub_download(REPO, "demo/pastis_40304.npz"))   # or demo/pastis_20350.npz
s2, s2_dates, s1, s1_dates, label = z["s2"], z["s2_dates"], z["s1"], z["s1_dates"], z["label"]
print(f"Sentinel-2: {s2.shape[0]} acquisitions {s2.shape[1:]}, {s2_dates[0]} .. {s2_dates[-1]}")
print(f"Sentinel-1: {s1.shape[0]} acquisitions {s1.shape[1:]}, {s1_dates[0]} .. {s1_dates[-1]}")

## Predict
`prepare` turns the raw acquisitions into monthly and seasonal composites (October 2018 to
September 2019 here) and 64 px tiles; `predict` runs MiSTR and stitches the tiles back.
`budget` sets where MiSTR adds fine-resolution detail (the share of uncertain 6 px windows; trained
at 0.5), and `seasons` how many seasons each selected window gets (1, 1.5 or 2; at 1.5 only the
more uncertain windows get a second season).

In [ ]:
x = prepare(s2, s2_dates, s1, s1_dates, start="2018-10")
out = model.predict(x, budget=0.5, seasons=2)
print(f"{out.coarse_tokens} coarse tokens + {out.fine_tokens} fine tokens; "
      f"{len(out.windows)} (window, month) pairs re-encoded")

In [ ]:
VOID = 19                                    # unlabelled pixels in PASTIS
cmap = ListedColormap(plt.cm.tab20(np.arange(20)))
def accuracy(pred):
    ok = label != VOID
    return (pred[ok] == label[ok]).mean()

summer = [i for i, d in enumerate(s2_dates) if d[5:7] in ("06", "07", "08")]
rgb = np.clip(np.median(s2[summer][:, [2, 1, 0]], 0).transpose(1, 2, 0) / 2500, 0, 1)

# where and when: each re-encoded window, coloured by the season its months come from
# (a window re-encoded in two seasons: left half the earlier season, right half the later)
SEASONS = ["Oct-Dec", "Jan-Mar", "Apr-Jun", "Jul-Sep"]
w = out.window_px
picked = {}
for month, y, x0 in out.windows.tolist():
    picked.setdefault((y, x0), set()).add(month // 3)
season = np.full(label.shape, -1)
for (y, x0), ss in picked.items():
    ss = sorted(ss)
    season[y:y + w, x0:x0 + w] = ss[0]
    if len(ss) > 1:
        season[y:y + w, x0 + w // 2:x0 + w] = ss[1]
season_colors = ["#8c564b", "#1f77b4", "#2ca02c", "#ffbf00"]

fig, ax = plt.subplots(1, 5, figsize=(20, 4.5))
panels = [(rgb, "Sentinel-2, summer median", None),
          (np.ma.masked_equal(label, VOID), "label", cmap),
          (out.coarse_logits.argmax(0).cpu().numpy(), f"coarse pass: {accuracy(out.coarse_logits.argmax(0).cpu().numpy()):.1%}", cmap),
          (out.classes.cpu().numpy(), f"MiSTR: {accuracy(out.classes.cpu().numpy()):.1%}", cmap)]
for a, (img, title, cm) in zip(ax, panels):
    a.imshow(img, cmap=cm, vmin=0 if cm else None, vmax=19 if cm else None, interpolation="nearest")
    a.set_title(title)
ax[4].imshow(rgb.mean(-1), cmap="gray", vmin=0, vmax=1)
ax[4].imshow(np.ma.masked_less(season, 0), cmap=ListedColormap(season_colors), vmin=0, vmax=3,
             alpha=0.85, interpolation="nearest")
ax[4].set_title("re-encoded at fine resolution\n(where, and which season)")
ax[4].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in season_colors], labels=SEASONS,
             loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
for a in ax:
    a.axis("off")
plt.show()
present = sorted(set(np.unique(label)) - {VOID})
print("classes in this patch:", ", ".join(f"{i} {names[i]}" for i in present))

## Trade accuracy for compute
Fewer seasons or lower budgets re-encode less; `budget=0` is the coarse pass alone.

In [ ]:
for seasons, budget in ((2, 0.5), (1.5, 0.5), (1, 0.5), (1, 0.25), (1, 0.1), (1, 0.0)):
    o = model.predict(x, budget=budget, seasons=seasons)
    print(f"seasons {seasons:3}, budget {budget:4}: {o.fine_tokens:5d} fine tokens, "
          f"pixel accuracy {accuracy(o.classes.cpu().numpy()):.1%}")

## Your own data
```python
x = prepare(s2, s2_dates, s1, s1_dates, start="2023-10")
```
- `s2`: (frames, 10, H, W) Sentinel-2 L2A bands `B02 B03 B04 B05 B06 B07 B08 B8A B11 B12`, reflectance x 10,000 (no BOA offset), 10 m
- `s1`: (frames, 2, H, W) Sentinel-1 `VV VH` in dB (trained on ascending passes)
- one date per frame; `start`: the first of 12 months; any H, W
- optional `s2_valid` / `s1_valid` (frames, H, W) to leave out clouds or nodata

The model was trained on French crop parcels (2018-2019, 19 classes); other regions, years or
crop types need fine-tuning. Code and paper: https://github.com/timgzhou/MiSTR

Sample data: PASTIS-R (Garnot et al., CC BY 4.0).